# Task 1: Repository and Kaggle Environment Setup Check (`align-setup`)

In [ ]:
!nvidia-smi
!ls -la /kaggle/input


In [ ]:
import os, shutil, subprocess, pathlib

# Locate the attached private align-code dataset under /kaggle/input
input_root = pathlib.Path('/kaggle/input')
candidates = list(input_root.rglob('submission_adapter.py'))
assert candidates, f'Could not find submission_adapter.py under {input_root}: {list(input_root.rglob("*"))}'
src_dir = candidates[0].parent
print('Found code dataset at:', src_dir)

work_repo = pathlib.Path('/kaggle/working/repo')
if work_repo.exists():
    shutil.rmtree(work_repo)
shutil.copytree(src_dir, work_repo)
os.chdir(work_repo)

commit_file = work_repo / 'COMMIT.txt'
if commit_file.exists():
    print('COMMIT:', commit_file.read_text().strip())


In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps


In [ ]:
import torch, transformers, datasets

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}:', torch.cuda.get_device_name(i))
print('Transformers version:', transformers.__version__)
print('Datasets version:', datasets.__version__)

# Run the public and edge-case adapter tests inside Kaggle
!PYTHONPATH=. python -m pytest -v tests/test_public_adapter_smoke.py tests/test_edge_cases.py
